In [1]:
import cv2
import gymnasium as gym
import ale_py
from gymnasium.wrappers import AtariPreprocessing, FrameStackObservation, RecordVideo

import numpy as np

import torch
import torch.nn as nn
from torch.utils.tensorboard import SummaryWriter

import time
import matplotlib.pyplot as plt

In [2]:
train_video_dir = f"../data/ppo/training"

In [3]:
class ResizeRender(gym.Wrapper):
    def render(self):
        frame = self.env.render()

        return cv2.resize(
            frame,
            (640, 840),
            interpolation=cv2.INTER_NEAREST
        )

In [4]:
train_episodes = 3000
training_period = train_episodes // 5
frameskips = 4

gym.register_envs(ale_py)
env = gym.make("ALE/MsPacman-v5", frameskip=1, render_mode="rgb_array")
env = AtariPreprocessing(env, scale_obs=True, frame_skip=frameskips, terminal_on_life_loss=True)
env = ResizeRender(env)
env = FrameStackObservation(env, stack_size=frameskips)
env = RecordVideo(env, video_folder=train_video_dir, name_prefix="training",
    episode_trigger=lambda x: x % training_period == 0)

c:\Users\vladi\AppData\Local\Programs\Python\Python312\Lib\site-packages\gymnasium\wrappers\rendering.py:292: UserWarning: WARN: Overwriting existing videos at d:\Study\Projects\pacman-agent\data\ppo\training folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(


In [5]:
action_size = env.action_space.n

In [6]:
device = "cuda" if torch.cuda.is_available() else "cpu"
# device = "cpu"
device

'cuda'

In [7]:
def layer_init(layer, std=np.sqrt(2), bias_const=0.0):
    torch.nn.init.orthogonal_(layer.weight, std)
    torch.nn.init.constant_(layer.bias, bias_const)
    return layer

class PacmanAgent(nn.Module):
    def __init__(self, n_hid, n_out):
        super().__init__()
        self.network = nn.Sequential(
            layer_init(nn.Conv2d(frameskips, 32, kernel_size=8, stride=4)),
            nn.ReLU(),
            
            layer_init(nn.Conv2d(32, 64, kernel_size=4, stride=2)),
            nn.ReLU(),         
               
            layer_init(nn.Conv2d(64, 64, kernel_size=3, stride=1)),
            nn.ReLU(),
            
            nn.Flatten(),
            layer_init(nn.Linear(64 * 7 * 7, n_hid)),
            nn.ReLU(),  
        )
        
        self.actor = layer_init(nn.Linear(n_hid, n_out), std=0.01)
        self.critic = layer_init(nn.Linear(n_hid, 1), std=1)                

    def get_value(self, x):
        return self.critic(self.network(x))

    def get_action_and_value(self, x, action=None):
        hidden = self.network(x)
        logits = self.actor(hidden)
        probs = torch.distributions.Categorical(logits=logits)
        if action is None:
            action = probs.sample()
        return action, probs.log_prob(action), probs.entropy(), self.critic(hidden)
    
def train(
    env,
    model,
    optimizer,
    episodes,
    update_epochs,
    num_steps,
    target_kl = 0.03,
    mini_batch_size = 64,
    gae_lambda = 0.95,
    clip_coef = 0.2,
    ent_coef = 0.01,
    vf_coef = 0.5,
    max_grad_norm = 0.5,
    norm_advantages = False,
    clip_vloss = False,
    lr_decay = True,
    gamma = 0.99,
    initial_lr = 1e-3,
    min_lr = 1e-5
):
    
    total_timesteps = episodes * num_steps
    
    writer = SummaryWriter(f"runs/pacman")
    
    batch_size = num_steps   
    
    obs = torch.zeros((num_steps, 1) + env.observation_space.shape).to(device)
    actions = torch.zeros((num_steps, 1) + env.action_space.shape, dtype=torch.long).to(device)
    logprobs = torch.zeros((num_steps, 1)).to(device)
    rewards = torch.zeros((num_steps, 1)).to(device)
    dones = torch.zeros((num_steps, 1)).to(device)
    values = torch.zeros((num_steps, 1)).to(device)
    
    global_step = 0
    start_time = time.time()
    
    for episode in range(1, episodes + 1):
        next_obs, _ = env.reset()
        next_obs = torch.FloatTensor(next_obs).to(device)
        next_done = torch.zeros(1).to(device)
        
        total_reward = 0
        
        if lr_decay:
            frac = max(
                0.0,
                1.0 - global_step / total_timesteps
            )

            lrnow = min_lr + frac * (initial_lr - min_lr)

            optimizer.param_groups[0]["lr"] = lrnow
        
        steps = 0
        done = False
        while steps < num_steps:
            if done: 
                next_obs, _ = env.reset()
                next_obs = torch.FloatTensor(next_obs).to(device)
                next_done = torch.zeros(1).to(device)
                done = False
            global_step += 1
            obs[steps] = next_obs
            dones[steps] = next_done
            
            with torch.no_grad():
                action, logprob, _, value = model.get_action_and_value(next_obs.unsqueeze(0))
                values[steps] = value.flatten()
            actions[steps] = action
            logprobs[steps] = logprob
            
            next_obs, reward, terminations, truncations, infos = env.step(action.cpu().numpy().item())
            next_done = np.logical_or(terminations, truncations)
            rewards[steps] = torch.tensor(reward).to(device).view(-1)
            total_reward += reward
            next_obs, next_done = torch.Tensor(next_obs).to(device), torch.tensor(next_done, dtype=torch.float32).to(device)
            done = truncations or terminations
            steps += 1
                        
        with torch.no_grad():
            next_value = model.get_value(next_obs.unsqueeze(0)).reshape(1, -1)
            advantages = torch.zeros_like(rewards).to(device)
            lastgaelam = 0
            for t in reversed(range(num_steps)):
                if t == num_steps - 1:
                    next_nonterminal = 1.0 - next_done
                    next_values = next_value
                else:
                    next_nonterminal = 1.0 - dones[t + 1]
                    next_values = values[t + 1]
                delta = rewards[t] + gamma * next_values * next_nonterminal - values[t]
                advantages[t] = lastgaelam = delta + gamma * gae_lambda * next_nonterminal * lastgaelam
            returns = advantages + values
         
        b_obs = obs.reshape((-1,) + env.observation_space.shape)
        b_logprobs = logprobs.reshape(-1)
        b_actions = actions.reshape((-1,) + env.action_space.shape)
        b_advanatages = advantages.reshape(-1)
        b_returns = returns.reshape(-1)
        b_values = values.reshape(-1)
        
        b_inds = np.arange(batch_size)
        clipfracs = []
        for epoch in range(update_epochs):
            np.random.shuffle(b_inds)
            for start in range(0, batch_size, mini_batch_size):
                end = start + mini_batch_size
                mb_inds = b_inds[start:end]
                
                _, newlogprob, entropy, newvalue = model.get_action_and_value(b_obs[mb_inds], b_actions.long()[mb_inds])
                logratio = newlogprob - b_logprobs[mb_inds]
                ratio = logratio.exp()
                
                with torch.no_grad():
                    old_approx_kl = (-logratio).mean()
                    approx_kl = ((ratio - 1) - logratio).mean()
                    clipfracs += [((ratio - 1.0).abs() > clip_coef).float().mean().item()]
                    
                mb_advantages = b_advanatages[mb_inds]
                if norm_advantages: 
                    mb_advantages = (mb_advantages-mb_advantages.mean()) / (mb_advantages.std() + 1e-8)
                
                pg_loss1 = -mb_advantages * ratio
                pg_loss2 = -mb_advantages * torch.clamp(ratio, 1 - clip_coef, 1 + clip_coef)
                pg_loss = torch.max(pg_loss1, pg_loss2).mean()
                
                new_value = newvalue.view(-1)

                if clip_vloss:
                    v_loss_unclipped = (
                        new_value - b_returns[mb_inds]
                    ) ** 2

                    v_clipped = (
                        b_values[mb_inds]
                        + torch.clamp(
                            new_value - b_values[mb_inds],
                            -clip_coef,
                            clip_coef,
                        )
                    )

                    v_loss_clipped = (
                        v_clipped - b_returns[mb_inds]
                    ) ** 2

                    v_loss_max = torch.max(
                        v_loss_unclipped,
                        v_loss_clipped
                    )

                    v_loss = 0.5 * v_loss_max.mean()

                else:
                    v_loss = 0.5 * (
                        (new_value - b_returns[mb_inds]) ** 2
                    ).mean()
                    
                entropy_loss = entropy.mean()
                loss = pg_loss - ent_coef * entropy_loss + v_loss * vf_coef
                
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
                optimizer.step()
                
            if target_kl is not None and approx_kl > target_kl:
                break
            
        y_pred, y_true = b_values.cpu().numpy(), b_returns.cpu().numpy()
        var_y = np.var(y_true)
        explained_var = np.nan if var_y == 0 else 1 - np.var(y_true - y_pred) / var_y
        
        writer.add_scalar("charts/learning_rate", optimizer.param_groups[0]["lr"], global_step)
        writer.add_scalar("losses/value_loss", v_loss.item(), global_step)
        writer.add_scalar("losses/policy_loss", pg_loss.item(), global_step)
        writer.add_scalar("losses/entropy", entropy_loss.item(), global_step)
        writer.add_scalar("losses/old_approx_kl", old_approx_kl.item(), global_step)
        writer.add_scalar("losses/approx_kl", approx_kl.item(), global_step)
        writer.add_scalar("losses/clipfrac", np.mean(clipfracs), global_step)
        writer.add_scalar("losses/explained_variance", explained_var, global_step)
        writer.add_scalar("charts/episode_reward", total_reward, global_step) 
        print(f"\rSPS: {int(global_step / (time.time() - start_time))} | episode: {episode} | reward: {total_reward} | global_step: {global_step}")
        writer.add_scalar("charts/SPS", int(global_step / (time.time() - start_time)), global_step)
            
    env.close()
    writer.close()

In [8]:
agent = PacmanAgent(512, action_size).to(device)
lr = 1e-4
optimizer = torch.optim.NAdam(agent.parameters(), lr=lr, eps=1e-5)

train(
    env=env,
    model=agent,
    optimizer=optimizer,
    episodes=train_episodes,
    
    update_epochs=6,
    num_steps=2048,
    target_kl=0.03,
    mini_batch_size=64,
    gae_lambda=0.95,
    clip_coef=0.2,
    ent_coef=0.01,
    vf_coef=0.5,
    max_grad_norm=0.5,
    
    norm_advantages=True,
    clip_vloss=True,
    lr_decay=True,
    
    gamma=0.99,
    initial_lr=lr,
)

SPS: 224 | episode: 1 | reward: 1220.0 | global_step: 2048
SPS: 259 | episode: 2 | reward: 1360.0 | global_step: 4096
SPS: 277 | episode: 3 | reward: 1260.0 | global_step: 6144
SPS: 289 | episode: 4 | reward: 1510.0 | global_step: 8192
SPS: 295 | episode: 5 | reward: 2380.0 | global_step: 10240
SPS: 302 | episode: 6 | reward: 1990.0 | global_step: 12288
SPS: 302 | episode: 7 | reward: 1290.0 | global_step: 14336
SPS: 302 | episode: 8 | reward: 3880.0 | global_step: 16384
SPS: 304 | episode: 9 | reward: 2020.0 | global_step: 18432
SPS: 306 | episode: 10 | reward: 2190.0 | global_step: 20480
SPS: 307 | episode: 11 | reward: 1750.0 | global_step: 22528
SPS: 308 | episode: 12 | reward: 2220.0 | global_step: 24576
SPS: 308 | episode: 13 | reward: 2050.0 | global_step: 26624
SPS: 310 | episode: 14 | reward: 2030.0 | global_step: 28672
SPS: 312 | episode: 15 | reward: 1960.0 | global_step: 30720
SPS: 312 | episode: 16 | reward: 1870.0 | global_step: 32768
SPS: 314 | episode: 17 | reward: 1760

In [9]:
torch.save(agent.state_dict(), "pacman-agent.pth")
torch.save(optimizer.state_dict(), "pacman-optimizer.pth")